# 365 Probabilidades · Dia #070
## Qual a probabilidade de um remédio que você sabe ser falso funcionar?

**Tipo:** Experimental
**Data de publicação:** 2026-08-22
**Ferramenta:** Python
**Decisão analisada:** Posso confiar numa melhora que só eu consigo medir?
**Hashtag:** #365Probabilidades #Dia070

---

### 📖 A História

O efeito placebo, do jeito que se conta, depende de engano. A pessoa acredita que
está tomando remédio, o corpo responde à crença, e a melhora aparece. Tira o engano
e some o efeito.

Essa história tem um problema óbvio: se ela fosse verdade, o placebo seria inútil
fora de um experimento, porque na vida real ninguém consegue enganar a si mesmo de
propósito.

Alguns pesquisadores resolveram testar a parte que parecia impossível. Entregaram um
frasco de comprimidos com uma explicação honesta: **isto é placebo, não tem princípio
ativo nenhum, e nós gostaríamos que você tomasse duas vezes ao dia mesmo assim.**

Chama-se placebo aberto. E o desconfortável é que funciona.

A pergunta interessante não é essa, porém. É outra: **funciona em quê?**

---

### 📚 O Conceito: dois tipos de desfecho

Num ensaio clínico existem duas famílias de medida.

O **autorrelato** é o que a pessoa diz: quanta dor sente, quão cansada está, como
avalia o próprio sono. É a medida que importa para quem vive dentro do corpo, e é
também a única disponível para boa parte do que a medicina trata.

O **desfecho objetivo** é o que um instrumento registra sem passar por você: pressão,
marcador inflamatório, velocidade de cicatrização, tempo real de sono.

As duas coisas se chamam "melhora" e são medidas diferentes. Quase toda meta-análise
mistura as duas num número só.

Este estudo separou. É a primeira vez que se testa **diretamente** se o efeito de
placebo aberto difere entre desfechos autorrelatados e objetivos.

E aqui cabe uma nota sobre o próprio projeto: o fator de correção de 0,80 que eu
aplico a proporções autorrelatadas existe justamente por essa desconfiança. Este dia
é o dia em que a desconfiança vira o objeto de estudo.

---

### 🧮 O Modelo

Duas meta-análises de ensaios randomizados, separando por tipo de desfecho.

**Fontes:**
- Fendel, J. C., Tiersch, C., Sölder, P., Gaab, J. & Schmidt, S., 2025 ·
  *Scientific Reports* · busca em oito bases até 9 de novembro de 2023 ·
  **60 ensaios randomizados, 63 comparações, n=4.554** · efeito geral
  **SMD = 0,35 [IC 95%: 0,26 · 0,44]**, p < 0,0001, I² = 53% ·
  **intervalo de predição de −0,17 a 0,87** ·
  autorrelato **k=55, n=3.919, SMD = 0,39** contra objetivo
  **k=17, n=1.250, SMD = 0,09** (Q = 7,24, p < 0,01) ·
  clínico **k=24, n=1.383, SMD = 0,47** contra não clínico
  **k=39, n=3.171, SMD = 0,29** (Q = 4,25, p < 0,05)
- Spille, L., Fendel, J. C., Seuling, P. D., Göritz, A. S. & Schmidt, S., 2023 ·
  *Scientific Reports* · amostras não clínicas · autorrelato **k=13, SMD = 0,43
  [0,28 · 0,58]** e objetivo **k=8, SMD = −0,02 [−0,25 · 0,21]**

**Nota metodológica sobre o fator ×0.80:** não se aplica. São tamanhos de efeito de
meta-análise de ensaios randomizados, não proporções de survey. O autorrelato aqui é
um dos braços comparados, não a fonte do dado agregado.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("Bibliotecas carregadas")

Bibliotecas carregadas


In [2]:
# --- DADOS DA LITERATURA ---
# Fendel, Tiersch, Solder, Gaab & Schmidt, 2025, Scientific Reports

k_geral, n_geral = 63, 4_554
n_ensaios        = 60
smd_geral        = 0.35
ic_geral         = (0.26, 0.44)
i2_geral         = 53          # heterogeneidade, em %
pred_geral       = (-0.17, 0.87)   # INTERVALO DE PREDICAO, nao de confianca

# Subgrupo por tipo de desfecho
k_auto, n_auto   = 55, 3_919
smd_auto         = 0.39
k_obj,  n_obj    = 17, 1_250
smd_obj          = 0.09
Q_desfecho, p_desfecho = 7.24, 0.01

# Subgrupo por populacao
k_clin, n_clin   = 24, 1_383
smd_clin         = 0.47
k_nclin, n_nclin = 39, 3_171
smd_nclin        = 0.29
Q_pop, p_pop     = 4.25, 0.05

# Spille, Fendel, Seuling, Goritz & Schmidt, 2023 - amostras nao clinicas
smd_auto_2023    = 0.43; ic_auto_2023 = (0.28, 0.58); k_auto_2023 = 13
smd_obj_2023     = -0.02; ic_obj_2023 = (-0.25, 0.21); k_obj_2023 = 8

aplica_fator_080 = False

print("=" * 70)
print("  DADOS - PLACEBO ABERTO: O QUE SE SENTE E O QUE SE MEDE")
print("=" * 70)
print(f"\n  Fendel et al., 2025 ({n_ensaios} ECRs, k={k_geral}, n={n_geral:,})"
      .replace(",", "."))
print(f"  -> Efeito geral: SMD = {smd_geral:.2f}"
      f"  IC 95% [{ic_geral[0]:.2f} · {ic_geral[1]:.2f}]  I2 = {i2_geral}%")
print(f"  -> INTERVALO DE PREDICAO: [{pred_geral[0]:.2f} · {pred_geral[1]:.2f}]")
print(f"     (o proximo estudo pode dar efeito nenhum)")
print(f"\n  Por tipo de desfecho:")
print(f"  -> Autorrelatado: k={k_auto}, n={n_auto:,}".replace(",", ".") +
      f"  SMD = {smd_auto:.2f}")
print(f"  -> Objetivo:      k={k_obj}, n={n_obj:,}".replace(",", ".") +
      f"  SMD = {smd_obj:.2f}")
print(f"  -> Diferenca entre os dois: Q = {Q_desfecho}, p < {p_desfecho}")
print(f"\n  Por populacao:")
print(f"  -> Clinica:     k={k_clin}, n={n_clin:,}".replace(",", ".") +
      f"  SMD = {smd_clin:.2f}")
print(f"  -> Nao clinica: k={k_nclin}, n={n_nclin:,}".replace(",", ".") +
      f"  SMD = {smd_nclin:.2f}")
print(f"  -> Diferenca: Q = {Q_pop}, p < {p_pop}")
print(f"\n  Spille et al., 2023 (nao clinicas), mesma fenda:")
print(f"  -> Autorrelatado: SMD = {smd_auto_2023:.2f}"
      f"  IC 95% [{ic_auto_2023[0]:.2f} · {ic_auto_2023[1]:.2f}]")
print(f"  -> Objetivo:      SMD = {smd_obj_2023:.2f}"
      f"  IC 95% [{ic_obj_2023[0]:.2f} · {ic_obj_2023[1]:.2f}]")
print(f"\n  Fator x0.80 aplicado: {aplica_fator_080}")
print("=" * 70)

  DADOS - PLACEBO ABERTO: O QUE SE SENTE E O QUE SE MEDE

  Fendel et al.. 2025 (60 ECRs. k=63. n=4.554)
  -> Efeito geral: SMD = 0.35  IC 95% [0.26 · 0.44]  I2 = 53%
  -> INTERVALO DE PREDICAO: [-0.17 · 0.87]
     (o proximo estudo pode dar efeito nenhum)

  Por tipo de desfecho:
  -> Autorrelatado: k=55. n=3.919  SMD = 0.39
  -> Objetivo:      k=17. n=1.250  SMD = 0.09
  -> Diferenca entre os dois: Q = 7.24, p < 0.01

  Por populacao:
  -> Clinica:     k=24. n=1.383  SMD = 0.47
  -> Nao clinica: k=39. n=3.171  SMD = 0.29
  -> Diferenca: Q = 4.25, p < 0.05

  Spille et al., 2023 (nao clinicas), mesma fenda:
  -> Autorrelatado: SMD = 0.43  IC 95% [0.28 · 0.58]
  -> Objetivo:      SMD = -0.02  IC 95% [-0.25 · 0.21]

  Fator x0.80 aplicado: False


In [3]:
# --- O MODELO ---
# Traducoes honestas de um tamanho de efeito padronizado.

# 1) Quanto do efeito geral sobra quando so se olham desfechos objetivos
proporcao_sobra = smd_obj / smd_auto
razao = smd_auto / smd_obj

# 2) Traducao do d de Cohen para linguagem de sobreposicao.
#    U3: proporcao do grupo controle que fica abaixo da MEDIA do grupo tratado.
#    Vale sob normalidade e variancias iguais, premissa declarada.
def u3(d):
    return stats.norm.cdf(d)


# 3) Probabilidade de superioridade: chance de uma pessoa sorteada do grupo
#    tratado ter resultado melhor que uma sorteada do controle.
def prob_superioridade(d):
    return stats.norm.cdf(d / np.sqrt(2))


# 4) Numero necessario para tratar, aproximacao de Furukawa a partir do d,
#    assumindo taxa de resposta de 20% no controle. Premissa declarada.
def nnt(d, taxa_controle=0.20):
    resposta_trat = stats.norm.cdf(d + stats.norm.ppf(taxa_controle))
    return 1 / (resposta_trat - taxa_controle)


print("=" * 70)
print("  MODELO - TRADUZINDO UM TAMANHO DE EFEITO")
print("=" * 70)
print(f"\n  1) A fenda entre o que se sente e o que se mede:")
print(f"  -> Autorrelatado {smd_auto:.2f} contra objetivo {smd_obj:.2f}")
print(f"  -> O efeito objetivo e {proporcao_sobra*100:.0f}% do autorrelatado")
print(f"  -> Ou seja, {razao:.1f}x menor")
print(f"\n  2) O que significa um d de {smd_geral:.2f}, em linguagem humana:")
for nome, d in [("geral", smd_geral), ("autorrelatado", smd_auto),
                ("objetivo", smd_obj)]:
    print(f"  -> {nome:<15} d = {d:.2f}"
          f" · P(um tratado ir melhor que um controle) = "
          f"{prob_superioridade(d)*100:.0f}%"
          f" · U3 = {u3(d)*100:.0f}%")
print(f"     Premissa declarada: normalidade e variancias iguais.")
print(f"     Sem essas premissas, d nao vira probabilidade.")
print(f"\n  3) Numero necessario para tratar (aproximacao de Furukawa,")
print(f"     assumindo 20% de resposta no grupo controle):")
for nome, d in [("geral", smd_geral), ("autorrelatado", smd_auto),
                ("objetivo", smd_obj)]:
    print(f"  -> {nome:<15} NNT = {nnt(d):.1f} pessoas")
print(f"     Premissa declarada. O artigo nao publica NNT.")
print(f"\n  4) E o numero que quase ninguem mostra:")
print(f"  -> IC 95% da media:      [{ic_geral[0]:.2f} · {ic_geral[1]:.2f}]")
print(f"  -> Intervalo de predicao: [{pred_geral[0]:.2f} · {pred_geral[1]:.2f}]")
print(f"  -> O primeiro fala sobre a MEDIA dos estudos.")
print(f"     O segundo fala sobre o PROXIMO estudo. E ele cruza o zero.")
print("=" * 70)

  MODELO - TRADUZINDO UM TAMANHO DE EFEITO

  1) A fenda entre o que se sente e o que se mede:
  -> Autorrelatado 0.39 contra objetivo 0.09
  -> O efeito objetivo e 23% do autorrelatado
  -> Ou seja, 4.3x menor

  2) O que significa um d de 0.35, em linguagem humana:
  -> geral           d = 0.35 · P(um tratado ir melhor que um controle) = 60% · U3 = 64%
  -> autorrelatado   d = 0.39 · P(um tratado ir melhor que um controle) = 61% · U3 = 65%
  -> objetivo        d = 0.09 · P(um tratado ir melhor que um controle) = 53% · U3 = 54%
     Premissa declarada: normalidade e variancias iguais.
     Sem essas premissas, d nao vira probabilidade.

  3) Numero necessario para tratar (aproximacao de Furukawa,
     assumindo 20% de resposta no grupo controle):
  -> geral           NNT = 9.0 pessoas
  -> autorrelatado   NNT = 8.0 pessoas
  -> objetivo        NNT = 38.3 pessoas
     Premissa declarada. O artigo nao publica NNT.

  4) E o numero que quase ninguem mostra:
  -> IC 95% da media:      [0.

In [4]:
# --- VISUALIZACAO ---

def br(n):
    return f"{n:,}".replace(",", ".")


DOURADO = '#c8a84b'
VERMELHO = '#c0392b'
VERDE = '#2a8a82'
CINZA = '#6b6a64'

# GRAFICO 1 - A fenda: o que se sente contra o que se mede
fig1, ax1 = plt.subplots(figsize=(12, 8))

rotulos = [f'AUTORRELATADO\nk={k_auto}, n={br(n_auto)}',
           f'OBJETIVO\nk={k_obj}, n={br(n_obj)}',
           '', f'Réplica 2023\nautorrelatado (k={k_auto_2023})',
           f'Réplica 2023\nobjetivo (k={k_obj_2023})']
valores = [smd_auto, smd_obj, np.nan, smd_auto_2023, smd_obj_2023]
ics = [None, None, None, ic_auto_2023, ic_obj_2023]
cores = [DOURADO, VERDE, None, DOURADO, VERDE]
y = np.arange(len(valores))[::-1]

for yi, val, ic, cor in zip(y, valores, ics, cores):
    if cor is None:
        continue
    ax1.scatter([val], [yi], s=300, color=cor, zorder=3)
    if ic is not None:
        ax1.plot([ic[0], ic[1]], [yi, yi], color=cor, linewidth=3)
        for lim in ic:
            ax1.plot([lim, lim], [yi - 0.09, yi + 0.09], color=cor, linewidth=3)
    ax1.text(val, yi + 0.24, f'{val:.2f}'.replace('.', ','), ha='center',
             fontsize=17, fontweight='bold', color=cor)

ax1.axvline(x=0, color='#333', linestyle='--', linewidth=1.6)
ax1.set_yticks(y)
ax1.set_yticklabels(rotulos, fontsize=11)
ax1.set_xlim(-0.35, 0.75)
ax1.set_ylim(-0.6, y.max() + 0.7)
ax1.set_xlabel('Diferença média padronizada (SMD)')
ax1.set_title('O efeito aparece onde você é quem mede\n'
              f'Fendel et al., 2025 ({n_ensaios} ECRs) e Spille et al., 2023',
              fontsize=14, pad=18)
ax1.text(0.5, -0.13,
         f'Diferença entre os dois tipos de desfecho em 2025: '
         f'Q = {Q_desfecho}, p < {p_desfecho}'.replace('.', ','),
         transform=ax1.transAxes, ha='center', fontsize=12, color=DOURADO,
         fontweight='bold')

plt.figtext(0.5, 0.005,
            'Fontes: Fendel et al., 2025 e Spille et al., 2023, Scientific Reports'
            '  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-070-grafico-01-fenda.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 1 salvo")

# GRAFICO 2 - Assinatura: intervalo de confianca contra intervalo de predicao
fig2, ax2 = plt.subplots(figsize=(12, 8))

x = np.linspace(-0.6, 1.2, 1000)
ep_pred = (pred_geral[1] - pred_geral[0]) / (2 * stats.norm.ppf(0.975))
densidade = stats.norm(smd_geral, ep_pred).pdf(x)

ax2.plot(x, densidade, color=DOURADO, linewidth=3)
ax2.fill_between(x, densidade, alpha=0.14, color=DOURADO)
dentro = (x >= pred_geral[0]) & (x <= pred_geral[1])
ax2.fill_between(x[dentro], densidade[dentro], alpha=0.25, color=DOURADO)

topo = densidade.max()
ax2.plot([ic_geral[0], ic_geral[1]], [topo * 1.12] * 2, color=VERDE, linewidth=6,
         solid_capstyle='butt')
ax2.scatter([smd_geral], [topo * 1.12], s=180, color=VERDE, zorder=4)
ax2.text(smd_geral, topo * 1.20, 'IC 95% da média: [0,26 · 0,44]', ha='center',
         fontsize=13, color=VERDE, fontweight='bold')

ax2.axvline(x=0, color=VERMELHO, linestyle='--', linewidth=2)
ax2.text(0.01, topo * 0.55, 'efeito\nnenhum', fontsize=12, color=VERMELHO)
for lim in pred_geral:
    ax2.axvline(x=lim, color=DOURADO, linestyle=':', linewidth=2)
ax2.text(smd_geral, topo * 0.08, 'intervalo de predição: [−0,17 · 0,87]',
         ha='center', fontsize=13, color='#7a5f1f', fontweight='bold')

ax2.set_ylim(0, topo * 1.42)
ax2.set_xlabel('Diferença média padronizada esperada no próximo estudo')
ax2.set_ylabel('Densidade')
ax2.set_title('A assinatura estatística: a média é confiável, o próximo estudo não é\n'
              'Intervalo de confiança fala da média. Intervalo de predição fala do que vem.',
              fontsize=14, pad=18)

plt.figtext(0.5, 0.005,
            'Fonte: Fendel et al., 2025, Scientific Reports  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-070-grafico-02-predicao.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 2 salvo")

# GRAFICO 3 - Traduzindo o d em linguagem humana
fig3, ax3 = plt.subplots(figsize=(12, 8))

x = np.linspace(-3.6, 3.6, 1000)
ax3.plot(x, stats.norm(0, 1).pdf(x), color=CINZA, linewidth=2.5, label='Controle')
ax3.fill_between(x, stats.norm(0, 1).pdf(x), alpha=0.13, color=CINZA)
ax3.plot(x, stats.norm(smd_geral, 1).pdf(x), color=DOURADO, linewidth=3,
         label=f'Placebo aberto (d = 0,35)')
ax3.fill_between(x, stats.norm(smd_geral, 1).pdf(x), alpha=0.18, color=DOURADO)

ax3.axvline(x=0, color=CINZA, linestyle=':', linewidth=1.6)
ax3.axvline(x=smd_geral, color=DOURADO, linestyle=':', linewidth=1.6)
ax3.annotate('', xy=(smd_geral, 0.43), xytext=(0, 0.43),
             arrowprops=dict(arrowstyle='<->', color='#333', lw=2))
ax3.text(smd_geral / 2, 0.445, 'd = 0,35', ha='center', fontsize=14,
         fontweight='bold', color='#333')

ax3.set_ylim(0, 0.52)
ax3.set_xlabel('Desfecho, em desvios padrão')
ax3.set_ylabel('Densidade')
ax3.legend(frameon=False, fontsize=13, loc='upper left')
ax3.set_title('O que um efeito de 0,35 quer dizer\n'
              f'{prob_superioridade(smd_geral)*100:.0f}% de chance de alguém do grupo '
              f'placebo ir melhor que alguém do controle',
              fontsize=14, pad=18)
ax3.text(0.5, -0.135,
         'Tradução válida sob normalidade e variâncias iguais. '
         'Premissa declarada, não publicada no artigo.',
         transform=ax3.transAxes, ha='center', fontsize=11, color=CINZA,
         style='italic')

plt.figtext(0.5, 0.005,
            'Fonte: Fendel et al., 2025, Scientific Reports  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-070-grafico-03-traducao.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 3 salvo")

Grafico 1 salvo
Grafico 2 salvo
Grafico 3 salvo


### 💡 O Insight

Sessenta ensaios randomizados. Quatro mil quinhentas e cinquenta e quatro pessoas.
Todas sabendo que estavam tomando placebo, porque isso foi dito na cara delas.

**E o efeito apareceu: 0,35, com intervalo de confiança de 0,26 a 0,44.**

Isso já derruba a versão popular do placebo. Ele não precisa de engano.

Mas o dia não é sobre isso. É sobre a segunda tabela do artigo.

Quando o desfecho é o que a pessoa **relata**, o efeito é 0,39. Quando é o que um
instrumento **mede**, cai para 0,09. Menos de um quarto. E a diferença entre os dois
grupos é estatisticamente significativa.

Numa meta-análise anterior, com amostras não clínicas, a mesma fenda apareceu ainda
mais nítida: 0,43 no autorrelato e −0,02 no objetivo. Zero, essencialmente.

Existem duas leituras honestas disso, e eu não sei escolher entre elas.

A primeira é desconfiada: o placebo aberto move a régua, não a doença. A pessoa passa
a relatar melhora porque foi convidada a esperar melhora.

A segunda é o contrário: sofrimento é subjetivo por natureza, e dizer que "só" o
autorrelato melhorou é desqualificar exatamente aquilo que a pessoa veio tratar. Dor
que dói menos é dor que dói menos, mesmo que o exame não mude.

As duas cabem nos mesmos números. Escolher entre elas não é estatística, é decidir o
que conta como melhora.

E tem um terceiro número, o menos comentado e o mais importante.

O intervalo de confiança do efeito médio vai de 0,26 a 0,44 e não chega perto do zero.
Mas o **intervalo de predição** vai de **−0,17 a 0,87**.

A diferença entre os dois é grande e quase ninguém explica: o intervalo de confiança
fala sobre a média dos estudos que já existem. O intervalo de predição fala sobre o
**próximo** estudo, ou sobre o próximo paciente. E esse cruza o zero.

Traduzindo: em média, funciona. No caso seguinte, pode não funcionar.

É a diferença entre "esse remédio funciona" e "esse remédio vai funcionar em você", e
a segunda frase nenhuma meta-análise consegue dizer.

*Que melhora sua, nos últimos meses, só existe na medida que você mesma faz?*

---

### ⚠️ Limitações do Modelo

- **Heterogeneidade moderada** (I² = 53%). Os ensaios diferem em condição tratada,
  instrução dada e desfecho medido.
- **O intervalo de predição inclui o zero.** O efeito médio é robusto; o efeito no
  próximo contexto não é garantido. Isso está no artigo e costuma ser omitido nas
  divulgações.
- **Os subgrupos são desiguais.** Autorrelato tem k=55 e n=3.919; objetivo tem k=17 e
  n=1.250. O braço objetivo é bem menor, portanto menos preciso, e parte da diferença
  pode ser precisão e não efeito.
- **Comparação entre subgrupos é observacional dentro da meta-análise.** Os estudos
  não foram sorteados para medir uma coisa ou outra, e quem escolheu desfecho objetivo
  pode ter estudado condições diferentes.
- **As traduções do d são minhas, não do artigo.** A probabilidade de superioridade, o
  U3 e o número necessário para tratar dependem de normalidade, variâncias iguais e,
  no caso do NNT, de uma taxa de resposta assumida de 20% no controle. São leituras de
  ordem de grandeza.
- Placebo aberto não é tratamento aprovado para nada, e nada aqui sugere substituir
  tratamento por comprimido inerte.
- Fator ×0.80 não aplicado: tamanhos de efeito de meta-análise de ensaios
  randomizados, não proporções de survey.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
